# Model Training

Document model decisions and test training snippets

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 04/04/2026   | Martin | CREATE   | Notebook created. Describe metrics selected | 
| 06/04/2026   | Martin | UPDATE   | Added data loading and model training components (testing script code) |
| 07/04/2026   | Martin | UPDATE   | Training loop testing |

# Content

* [Introduction](#introduction)
* [Metrics Used](#metrics-used)
* [Data Loading](#data-loading)

# Introduction

Board game recommendation system using the two-tower architecture with a reranker.

# Metrics Used

- __Recall@K__ - Information retrieval and recommendation metrics that measures the proportion of relevant items found in the top-K results. Results here are not rank aware i.e does not matter which position in the top-K the item was found in, as long as it's inside there
$$
\frac{Relevant\ Items\ in\ Top\ K}{Total\ Relevant\ Items}
$$

- __Normalized Discounted Cumulative Gains (NDCG@K)__ - Measures the gain of an item based on its relevantce grade and position in the top-K results. It rewards placing highly relevant items at the top of the list
  * $rel_i$: Relevance score (e.g whether the item showed up in the top-K spots)
$$
NDCG@K = \frac{DCG@K}{IDCG@K} \\
DCG@K = \sum_{k=1}^{K}\frac{rei_i}{log_2(i+1)} \\
IDCG@K = Maximum\ achievable\ DCG
$$
- __Mean Reciprocal Rank (MRR)__ - Ranking quality metrics that considers the position of the first relevant item in the ranked list. Range from 0-1 where "1" is the first relevant item being at the top of the list
  * $U$: Total number of users
$$
MRR = \frac{1}{U}\sum_{u=1}{U}\frac{1}{rank_i}
$$
- __HitRate@K__ - Measures the number of users whom at least one relevant item is present in the top-K list. Coarse metric
- __Mean Average Precision (MAP@K)__ - Ranking quality metric that considers the number of relevant recommendations and their position in the list.
  * $N$: Total number of relevant items in the top-K results
  * Divide the total number of relevant items by the total number of items until this position
$$
MAP@K = \frac{1}{U}\sum_{u=1}^U AP@K_u \\
AP@K = \frac{1}{N}\sum_{k=1}^K Precision(k) \times rel(k)
$$

- __Coverage@K__ - Measures the percentage of unique items that was recommended over the total number of items. Evalutes the diversity of items
$$
Coverage@K = \frac{Number\ of\ unique\ items\ recommended\ in\ top-K}{Total\ number\ of\ items\ in\ catalog}
$$

# Data Loading

## Converting to DuckDB calls

Original code uses CSVs, converting to DuckDB calls

In [1]:
import duckdb

In [2]:
DUCKDB_PATH = "../data/bgg_db.duckdb"

In [3]:
con = duckdb.connect(DUCKDB_PATH)

In [4]:
con.sql("SHOW ALL TABLES")

┌──────────┬─────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │   name   │                                                                                                                                                             column_names                                                                                                                                                       

In [5]:
games = con.sql("SELECT * FROM bgg.games").fetchdf()
comments = con.sql("SELECT * FROM bgg.comments").df()

In [11]:
games['id']

0     342942
1     161936
2     174430
3     316554
4     397598
5     233078
6     115746
7     167791
8     187645
9     162886
10    291457
11    220308
12     12333
13     84876
14    182028
Name: id, dtype: int32

In [5]:
con.sql("SELECT DISTINCT id FROM bgg.games;")

┌────────┐
│   id   │
│ int32  │
├────────┤
│ 182028 │
│ 115746 │
│ 295770 │
│ 161936 │
│ 174430 │
│ 162886 │
│ 220308 │
│ 291457 │
│  12333 │
│ 418059 │
│ 316554 │
│ 342942 │
│ 233078 │
│ 187645 │
│ 397598 │
│  84876 │
│ 338960 │
│ 193738 │
│ 421006 │
│ 167791 │
└────────┘

In [6]:
con.close()

## Testing Supabase calls

In [7]:
import os
from dotenv import load_dotenv
from supabase import create_client, Client
import pandas as pd
load_dotenv()

True

In [8]:
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)

In [9]:
response = (
  supabase.table("comp_game_log")
  .select("*")
  .execute()
)

In [10]:
records = pd.DataFrame(response.data)

In [5]:
records.head()

,id,session_id,created_at,date_played,game_id,game_title,game_weight,game_length,num_players,profile_id,...,win_contrib,score,high_score,quarter,month,year,is_tie,is_first_play,rating,created_by
0,175,92697a9182a3,2026-01-24T08:46:49.863284+00:00,2026-01-24,118048,Targi,2.3390,60,2,10,...,100,4.33649,False,1,0,2026,False,True,NaN,10
1,176,92697a9182a3,2026-01-24T08:46:49.863284+00:00,2026-01-24,118048,Targi,2.3390,60,2,11,...,0,1.08412,False,1,0,2026,False,True,NaN,10
2,196,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,11,...,250,5.95074,False,1,1,2026,True,False,NaN,11
3,197,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,21,...,0,1.48768,False,1,1,2026,True,True,NaN,11
4,198,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,22,...,0,0.66119,False,1,1,2026,False,True,NaN,11


In [11]:
records['game_id'].unique()

array([118048, 393429, 143693, 341945, 358661, 414317,  54043, 200147,
       316554, 286749,    172, 397598, 314582, 266810, 350933, 312484,
       315560, 420361, 256960, 317985, 367966, 196340, 366013, 249670,
        30869, 378979, 414685, 319966, 217372, 201808,  96913, 321608,
       291453, 342942, 418470, 363622,   6249, 237182, 350184, 281259,
       359871, 400314, 350198, 366251, 418062, 295947, 385761, 231733,
       331106, 106217, 365258, 347703, 265736, 207330, 183251, 224517,
       436126, 425005,    811, 338376])

In [13]:
records[records['game_id'] == 393429]

,id,session_id,created_at,date_played,game_id,game_title,game_weight,game_length,num_players,profile_id,...,win_contrib,score,high_score,quarter,month,year,is_tie,is_first_play,rating,created_by
2,196,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,11,...,250,5.95074,False,1,1,2026,True,False,NaN,11
3,197,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,21,...,0,1.48768,False,1,1,2026,True,True,NaN,11
4,198,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,22,...,0,0.66119,False,1,1,2026,False,True,NaN,11
5,200,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,48,...,0,0.23803,False,1,1,2026,False,True,NaN,11
6,199,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,23,...,0,0.37192,False,1,1,2026,False,True,5.0,11
63,136,8760ce89cf5f,2026-01-09T16:56:28.530074+00:00,2026-01-09,393429,Critter Kitchen,2.4595,60,5,36,...,250,5.95991,False,1,0,2026,False,True,NaN,11
64,137,8760ce89cf5f,2026-01-09T16:56:28.530074+00:00,2026-01-09,393429,Critter Kitchen,2.4595,60,5,19,...,0,5.95991,False,1,0,2026,False,True,NaN,11
65,138,8760ce89cf5f,2026-01-09T16:56:28.530074+00:00,2026-01-09,393429,Critter Kitchen,2.4595,60,5,11,...,0,0.66221,False,1,0,2026,False,True,NaN,11
66,139,8760ce89cf5f,2026-01-09T16:56:28.530074+00:00,2026-01-09,393429,Critter Kitchen,2.4595,60,5,37,...,0,0.37249,False,1,0,2026,False,True,NaN,11
67,140,8760ce89cf5f,2026-01-09T16:56:28.530074+00:00,2026-01-09,393429,Critter Kitchen,2.4595,60,5,38,...,0,0.23840,False,1,0,2026,False,True,NaN,11


In [ ]:
records.columns

Index(['id', 'session_id', 'created_at', 'date_played', 'game_id',
       'game_title', 'game_weight', 'game_length', 'num_players', 'profile_id',
       'group_id', 'is_vp', 'victory_points', 'is_winner', 'position',
       'win_contrib', 'score', 'high_score', 'quarter', 'month', 'year',
       'is_tie', 'is_first_play', 'rating', 'created_by'],
      dtype='str')

In [21]:
games.head()

,id,name,description,year_published,min_players,max_players,suggested_players,playing_time,min_playing_time,max_playing_time,...,family_ids,families,designer_ids,designers,artist_ids,artists,num_ratings,rating,bayes_rating,complexity
0,342942,Ark Nova,"In Ark Nova, you will plan and design a modern...",2021,1,4,2,150,90,150,...,"[110926, 67874, 27383, 9470, 50153, 61646, 616...","[Animals: Okapi, Components: Hexagonal Tiles, ...",[138517],[Mathias Wigge],"[138547, 11462, 12484]","[Steffen Bieker, Loïc Billiau, Dennis Lohausen]",60007,8.54188,8.35466,3.7957
1,161936,Pandemic Legacy: Season 1,Pandemic Legacy is a co-operative campaign gam...,2015,2,4,4,60,60,60,...,"[64952, 65191, 3430, 24281, 25404, 61854, 7222...","[Components: Map (Global Scale), Components: M...","[442, 378]","[Rob Daviau, Matt Leacock]",[14057],[Chris Quilliams],57179,8.50451,8.34821,2.8283
2,174430,Gloomhaven,Gloomhaven is a game of Euro-inspired tactica...,2017,1,4,3,120,60,120,...,"[59218, 67953, 25158, 65191, 66335, 68438, 700...","[Category: Dungeon Crawler, Components: Map (C...",[69802],[Isaac Childres],"[77084, 173812, 78961, 84269]","[Alexandr Elichev, Lucile Mathieu, Josh T. McD...",66872,8.54235,8.30157,3.9182
3,316554,Dune: Imperium,Dune: Imperium is a game that uses deck buildi...,2020,1,4,3,120,60,120,...,"[5838, 79006, 65191, 104952, 112176, 78432, 77...","[Books: Dune, Components: Meeples, Components...",[92370],[Paul Dennen],"[119957, 155730, 73773, 31001]","[Clay Brooks, Brett Nienburg, Raul Ramos, Nate...",57700,8.41386,8.22361,3.0757
4,397598,Dune: Imperium – Uprising,"In Dune: Imperium Uprising, you want to contin...",2023,1,6,4,120,60,120,...,"[5838, 79009, 59705, 78198, 5666]","[Books: Dune, Game: Dune: Imperium, Misc: Long...",[92370],[Paul Dennen],"[119957, 115889, 73773, 31001]","[Clay Brooks, Derek Herring, Raul Ramos, Nate ...",17564,8.70430,8.22709,3.5244


# Model Training

_Contrastive Learning:_ Classification of data instances based on their similarities; Positions analogous instances in close procimity within a latent space, concurently ensuring a distinct separation of dissimilar ones.

> _"Instances exhibiting similarity should be closely aligned in the learned embedding space, whereas those that are dissimilar should be positioned further apart."_

- In unsupervised learning, it leverages properties of dat to generate pseudo-labels
- Quantification of similarity between instances typically employs distance metrics

_Information Noise-Contrastive Estimation (InfoNCE):_ Loss function designed for unsupervised learning. 

In [14]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append(os.path.abspath("../src"))

In [23]:
from preprocessing import load_data, multi_hot_encode
from config import DataConfig, EmbeddingConfig, GAME_NUMERIC_FEATURES, GAME_TAG_COLUMNS
from preprocessing import (
  load_data,
  build_tag_vocabularies,
  fit_numeric_scaler,
  scale_numeric_features
)
from game_builder import build_game_profiles, get_profile_dim
import numpy as np
from user_builder import build_all_user_profiles, get_user_feature_dim

In [24]:
sum([len(v) for k, v in tag_vocabs.items()])

360

In [29]:
len(tag_vocabs['families'])

119

In [17]:
games, records, comments = load_data(DataConfig())
print(f"    {len(games)} games, {len(records)} records, {len(comments)} comments")

    21 games, 298 records, 2168 comments


In [24]:
emb_cfg = EmbeddingConfig()
tag_vocabs = build_tag_vocabularies(games)
scaler = fit_numeric_scaler(games)
numeric_features = scale_numeric_features(games, scaler)
game_profiles, text_encoder = build_game_profiles(
  games, comments, tag_vocabs, numeric_features, emb_cfg
)
profile_dim = get_profile_dim(tag_vocabs, emb_cfg, len(GAME_NUMERIC_FEATURES))

[GAME BUILDER] Using Sentence transformer


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [26]:
user_features = build_all_user_profiles(records, game_profiles, profile_dim)
user_dim = get_user_feature_dim(profile_dim)

In [28]:
user_dim

1152

In [20]:
from pathlib import Path

In [ ]:
%load_ext watermark
%watermark